# Step through next-token prediction

This notebook runs the loop from the [Next-token prediction](https://bradleyboehmke.github.io/ai-engineering-notes/dictionary/next-token-prediction.html) dictionary entry on GPT-2, a small open model with a 50,257-token vocabulary. It runs on a free Colab CPU, and GPT-2 downloads (about 500 MB) the first time.

There are three parts:

1. **One step.** Enter some text and see the 10 tokens the model thinks are most likely to come next, with their probabilities.
2. **The loop.** Enter some text and a number of tokens, and see the text the loop produces.
3. **The path.** See every step of that loop: which token was chosen, how likely it was, and what it beat.

Every choice here is greedy: the loop always takes the most likely token. Real LLM products usually sample instead, so they won't always pick the top token.

## 1. Install the libraries

Colab already has both, so this is quick there.

In [ ]:
%pip install --quiet torch transformers pandas

## 2. The code

This is the same loop as [`step.py`](https://github.com/bradleyboehmke/ai-engineering-notes/blob/main/learning/llm-mechanics/experiments/step.py), split into two functions so the cells below can reuse it.

- `next_token_probs` runs the model once. It returns the `k` most likely next tokens and their probabilities.
- `generate` runs the loop: score, convert with softmax, choose the top token, append it, repeat. It records every step so you can see the path afterward.

`␣` marks a leading space, which is part of the token.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2")
model.eval()

def show(token):
    """Make a token readable: mark a leading space and newlines."""
    return token.replace(" ", "␣", 1) if token.startswith(" ") else token.replace("\n", "↵")

def next_token_probs(ids, k=10):
    """One step: run the model once and return the k most likely next tokens."""
    with torch.no_grad():
        logits = model(ids).logits[0, -1]    # one score per vocabulary token
    probs = torch.softmax(logits, dim=-1)     # scores -> probability distribution
    top = torch.topk(probs, k)
    return [(i.item(), tok.decode(i), p.item()) for p, i in zip(top.values, top.indices)]

def generate(text, n_tokens, k=5):
    """The loop: choose the top token, append it, and repeat n_tokens times."""
    ids = tok(text, return_tensors="pt").input_ids
    path = []
    for step in range(1, n_tokens + 1):
        top = next_token_probs(ids, k)
        next_id, token, p = top[0]            # greedy: take the most likely
        path.append({"step": step, "token": token, "probability": p, "alternatives": top[1:]})
        ids = torch.cat([ids, torch.tensor([[next_id]])], dim=1)  # append, then loop
        if next_id == tok.eos_token_id:       # end-of-text token: stop early
            break
    return tok.decode(ids[0]), path

## 3. One step: the top 10 next tokens

Enter any text and run the cell. With the default text you should see `␣$` at about 21.5%, as on the term page.

The probabilities don't add up to 100% because these are only 10 of 50,257 tokens. The rest share whatever is left.

In [ ]:
text = "The quarterly revenue increased by"  # @param {type:"string"}

import pandas as pd

ids = tok(text, return_tensors="pt").input_ids
top10 = next_token_probs(ids, k=10)
table = pd.DataFrame(
    [(rank, show(token), p) for rank, (_, token, p) in enumerate(top10, 1)],
    columns=["rank", "token", "probability"],
).set_index("rank")

print(f"Top 10 cover {table.probability.sum():.1%}; the other {len(tok) - 10:,} tokens share the rest.")
table.style.format({"probability": "{:.1%}"}).bar(subset=["probability"], vmin=0, vmax=1, color="#9ec5fe")

## 4. The loop: generate some tokens

Enter the starting text and how many tokens to add. With the defaults you should get the same result as the term page: *The quarterly revenue increased by \$1.3 billion to \$1*.

The loop stops early if the model chooses the end-of-text token.

In [ ]:
text = "The quarterly revenue increased by"  # @param {type:"string"}
n_tokens = 8  # @param {type:"integer"}

result, path = generate(text, n_tokens)
print(f"{len(path)} passes, {len(path)} new tokens:\n")
print(result)

## 5. The path: how the loop got there

Run this after step 4. The first view shades each generated token by how likely it was when it was chosen: dark means the model was nearly certain, pale means a close call. Hover over a token to see what it beat.

The table shows the same path with the runner-up at each step. Look for the rows where the margin is small. Those near-ties are where a different choice would have sent the rest of the text somewhere else.

In [ ]:
from html import escape
from IPython.display import HTML, display

def chip(entry):
    p = entry["probability"]
    alts = ", ".join(f"{show(t)} {q:.1%}" for _, t, q in entry["alternatives"])
    text_color = "white" if p > 0.5 else "black"
    return (f'<div style="display:inline-block;text-align:center;margin:2px">'
            f'<div title="Step {entry["step"]}. Beat: {escape(alts)}" '
            f'style="background:rgba(39,128,227,{0.12 + 0.88 * p:.2f});color:{text_color};'
            f'font-family:monospace;padding:4px 8px;border-radius:6px;white-space:pre">{escape(show(entry["token"]))}</div>'
            f'<div style="font-size:11px;color:#6c757d">{p:.0%}</div></div>')

prompt = (f'<div style="display:inline-block;margin:2px;vertical-align:top;font-family:monospace;'
          f'padding:4px 8px;border:1px solid #adb5bd;border-radius:6px">{escape(text)}</div>')
display(HTML(prompt + "".join(chip(e) for e in path)))

steps = pd.DataFrame([{
    "step": e["step"],
    "chosen": show(e["token"]),
    "probability": e["probability"],
    "runner-up": show(e["alternatives"][0][1]),
    "runner-up probability": e["alternatives"][0][2],
    "margin (points)": 100 * (e["probability"] - e["alternatives"][0][2]),
} for e in path]).set_index("step")

steps.style.format({"probability": "{:.1%}", "runner-up probability": "{:.1%}", "margin (points)": "{:.1f}"})

## What to look for

- **The top token often has well under 50%.** The model's output is a whole distribution, and greedy decoding throws most of it away.
- **Small margins matter.** With the defaults, step 4 chooses `3` over `4` by 0.7 points, and every token after it is built on that choice.
- **Numbers are made up.** Nothing in *The quarterly revenue increased by* says \$1.3 billion. Try putting the real figures in your text and see whether the model uses them.
- **Results can differ slightly on another machine.** A GPU or a different library version can shift the last decimal, and occasionally flip a near-tie.